In [18]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Leitura direta do dataset a partir do endereço fornecido
url = "https://raw.githubusercontent.com/Fer412/Dataset_2026/refs/heads/main/heart1.csv"
data = pd.read_csv(url, sep=';')

print("Dados Iniciais:")
print(data.head())

# 2. Tratamento de Missing Values (Imputação)
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = data.select_dtypes(include=['object']).columns

if len(num_cols) > 0:
    imputer_num = SimpleImputer(strategy='mean')
    data[num_cols] = imputer_num.fit_transform(data[num_cols])

if len(cat_cols) > 0:
    imputer_cat = SimpleImputer(strategy='most_frequent')
    data[cat_cols] = imputer_cat.fit_transform(data[cat_cols])

# 3. Tratamento de valores não numéricos (Atribuição Ordinal)
if len(cat_cols) > 0:
    encoder = OrdinalEncoder()
    data[cat_cols] = encoder.fit_transform(data[cat_cols])

# Gravar o novo dataset com o nome exigido
data.to_csv("heart_clean_1.csv", index=False)
print("\nAtividade 1 concluída. Ficheiro 'heart_clean_1.csv' guardado.")

Dados Iniciais:
   age sex  cp  trtbps  chol  fbs  restecg  thalachh  exng  oldpeak  slp  caa  \
0   63   M   3     145   233    1        0       150     0      2.3    0    0   
1   37   M   2     130   250    0        1       187     0      3.5    0    0   
2   41   F   1     130   204    0        0       172     0      1.4    2    0   
3   56   M   1     120   236    0        1       178     0      0.8    2    0   
4   57   F   0     120   354    0        1       163     1      0.6    2    0   

   thall  output  
0      1       1  
1      2       1  
2      2       1  
3      2       1  
4      2       1  

Atividade 1 concluída. Ficheiro 'heart_clean_1.csv' guardado.


In [19]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Carregar o dataset limpo
data_clean = pd.read_csv("heart_clean_1.csv")

# Separar os atributos (X) da variável dependente (y)
X_features = data_clean.iloc[:, :-1].columns

# Aplicar o StandardScaler nas características
sc = StandardScaler()
data_clean[X_features] = sc.fit_transform(data_clean[X_features])

# Gravar o resultado no ficheiro exigido
data_clean.to_csv("heart_scale_2.csv", index=False)
print("Atividade 2 concluída. Ficheiro 'heart_scale_2.csv' guardado.")

Atividade 2 concluída. Ficheiro 'heart_scale_2.csv' guardado.


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Carregar o dataset normalizado
data_scale = pd.read_csv("heart_scale_2.csv")

# Divisão em conjuntos de treino e teste (80% treino, 20% teste)
training_set, validation_set = train_test_split(data_scale, test_size=0.2, random_state=21)

# Gravar com as designações estipuladas no enunciado
training_set.to_csv("heart_treino_3.csv", index=False)
validation_set.to_csv("heart_teste_3.csv", index=False)

print("Atividade 3 concluída. Ficheiros 'heart_treino_3.csv' e 'heart_teste_3.csv' guardados.")

Atividade 3 concluída. Ficheiros 'heart_treino_3.csv' e 'heart_teste_3.csv' guardados.


In [21]:
import pandas as pd
from sklearn.neural_network import MLPClassifier
import pickle

# Carregar os dados específicos de treino
data_train = pd.read_csv("heart_treino_3.csv")

# Extração de atributos (X) e rótulos (y)
X_train = data_train.iloc[:, :-1].values
Y_train = data_train.iloc[:, -1].values

# Inicialização e treino do classificador respeitando as especificações
classifier = MLPClassifier(
    hidden_layer_sizes=(150, 100, 50),
    max_iter=100000,
    activation='relu',
    solver='adam',
    random_state=1
)

# Ajustar o modelo aos dados de treino
classifier.fit(X_train, Y_train)

# Guardar o modelo treinado para utilização futura
filename = 'predictorHeart.sav'
pickle.dump(classifier, open(filename, 'wb'))

print("Atividade 4 concluída. Rede Neuronal Artificial treinada e modelo 'predictorHeart.sav' guardado.")

Atividade 4 concluída. Rede Neuronal Artificial treinada e modelo 'predictorHeart.sav' guardado.


In [22]:
import pandas as pd
import pickle
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Carregar o modelo guardado
filename = 'predictorHeart.sav'
loaded_model = pickle.load(open(filename, 'rb'))

# 2. Carregar os dados conhecidos de teste (com as respostas corretas para avaliar)
data_test = pd.read_csv("heart_teste_3.csv")
X_test = data_test.iloc[:, :-1].values
y_test = data_test.iloc[:, -1].values

# 3. Executar o Teste do modelo
y_test_pred = loaded_model.predict(X_test)

# 4. Avaliação estatística do desempenho
cm = confusion_matrix(y_test, y_test_pred)
acc = accuracy_score(y_test, y_test_pred)

print("\n--- RESULTADOS DO TESTE DE VALIDAÇÃO ---")
print("Matriz de Confusão:")
print(cm)
print(f"\nExatidão global (Accuracy): {acc * 100:.2f}%")
print(f"Traço da Matriz de Confusão (Previsões corretas): {cm.trace()}")
print(f"Total de observações testadas: {cm.sum()}")


--- RESULTADOS DO TESTE DE VALIDAÇÃO ---
Matriz de Confusão:
[[26  6]
 [ 9 20]]

Exatidão global (Accuracy): 75.41%
Traço da Matriz de Confusão (Previsões corretas): 46
Total de observações testadas: 61


In [23]:
import pandas as pd
import pickle

# 1. Carregar o modelo treinado e validado
filename = 'predictorHeart.sav'
loaded_model = pickle.load(open(filename, 'rb'))

# 2. Receber dados de novos pacientes (Simulação: sem a coluna 'output')
dados_novos_pacientes = pd.read_csv("heart_teste_3.csv").iloc[:, :-1]

# 3. Executar a Previsão pura (Descobrir o resultado clínico)
previsoes_finais = loaded_model.predict(dados_novos_pacientes.values)

# 4. Associar os resultados aos respetivos novos pacientes
dados_novos_pacientes['Diagnostico_Previsto'] = previsoes_finais

print("--- PREVISÕES EXECUTADAS PARA NOVOS PACIENTES ---")
print(dados_novos_pacientes[['age', 'sex', 'cp', 'Diagnostico_Previsto']].head())

--- PREVISÕES EXECUTADAS PARA NOVOS PACIENTES ---
        age       sex        cp  Diagnostico_Previsto
0 -0.702136  0.681005 -0.938515                   0.0
1  0.841908 -1.468418  1.002577                   1.0
2 -1.805024  0.681005  1.973123                   1.0
3  0.290464  0.681005 -0.938515                   0.0
4  1.724218  0.681005 -0.938515                   0.0
